In [1]:
!pip install -q transformers peft trl bitsandbytes accelerate datasets

import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Không có GPU — vào Runtime > Change runtime type > GPU'}")

GPU available: True
GPU name: Tesla T4


In [2]:
!pip install -U "bitsandbytes>=0.46.1" accelerate

In [18]:
# Load dataset RAFT đã tạo
import copy
import json
import os
import random
from collections import Counter
from sklearn.model_selection import StratifiedGroupKFold

DATA_PATH = "/content/drive/MyDrive/raft_project/raft_checkpoint_chunks"

raft_samples = []

for filename in sorted(os.listdir(DATA_PATH)):
    if filename.endswith(".json"):
        filepath = os.path.join(DATA_PATH, filename)
        with open(filepath, "r", encoding="utf-8") as f:
            try:
                payload = json.load(f)
                raft_samples.extend(payload.get("samples", []))
            except Exception as e:
                print(f"Warning: Lỗi đọc file {filename}: {e}")

# Lọc an toàn các sample hợp lệ
raft_samples = [
    s for s in raft_samples
    if s.get("source_chunk_id") is not None
    and s.get("answer_mode") in {"memory", "grounded"}
]

# Chia theo source_chunk_id để các sample cùng chunk luôn ở cùng một split
# Stratify theo answer_mode để giữ tỷ lệ memory/grounded ổn định giữa 2 tập
sample_indices = list(range(len(raft_samples)))
labels = [sample["answer_mode"] for sample in raft_samples]
groups = [sample["source_chunk_id"] for sample in raft_samples]

splitter = StratifiedGroupKFold(
    n_splits=7,       # ~15% test
    shuffle=True,
    random_state=42,
)

train_indices, test_indices = next(
    splitter.split(sample_indices, labels, groups)
)

train_samples_raw = [raft_samples[i] for i in train_indices]
test_samples = [raft_samples[i] for i in test_indices]

# Kiểm tra chặt chẽ tính độc lập của tập train và test
train_chunk_ids = {sample["source_chunk_id"] for sample in train_samples_raw}
test_chunk_ids = {sample["source_chunk_id"] for sample in test_samples}

assert train_chunk_ids.isdisjoint(test_chunk_ids), "LỖI: Rò rỉ source_chunk_id giữa train và test!"
assert len(train_samples_raw) + len(test_samples) == len(raft_samples)

# -------------------------------------------------------------------------
# OVERSAMPLING: Nhân đôi Memory trên tập Train (Dùng Deep Copy & Cập nhật ID)
# -------------------------------------------------------------------------
grounded_train = [s for s in train_samples_raw if s["answer_mode"] == "grounded"]
memory_train = [s for s in train_samples_raw if s["answer_mode"] == "memory"]

# Tạo bản sao sâu (deep copy) và đánh dấu id mới để tránh xung đột object
duplicated_memory = []
for s in memory_train:
    dup_s = copy.deepcopy(s)
    dup_s["id"] = f"{s['id']}_dup"
    duplicated_memory.append(dup_s)

# Gộp: 1x Grounded + 2x Memory
train_samples = grounded_train + memory_train + duplicated_memory

# Xáo trộn thứ tự dữ liệu train
rng = random.Random(42)
rng.shuffle(train_samples)

# Báo cáo kết quả
print("==================================================")
print(f"📂 Tổng sample hợp lệ: {len(raft_samples)}")
print(f"🔹 Train chunks: {len(train_chunk_ids)} | Test chunks: {len(test_chunk_ids)}")
print(f"🔹 Chunk overlap: {len(train_chunk_ids & test_chunk_ids)} (Phải bằng 0)")
print("--------------------------------------------------")
print("Phân bố trước oversampling:")
print(f"  Train raw : {Counter(s['answer_mode'] for s in train_samples_raw)}")
print(f"  Test      : {Counter(s['answer_mode'] for s in test_samples)}")
print("--------------------------------------------------")
print("Phân bố sau oversampling:")
print(f"  Train final: {Counter(s['answer_mode'] for s in train_samples)}")
print(f"  Tổng mẫu train: {len(train_samples)} (tăng thêm {len(duplicated_memory)} mẫu memory)")
print("==================================================")

📂 Tổng sample hợp lệ: 1734
🔹 Train chunks: 742 | Test chunks: 125
🔹 Chunk overlap: 0 (Phải bằng 0)
--------------------------------------------------
Phân bố trước oversampling:
  Train raw : Counter({'grounded': 1031, 'memory': 453})
  Test      : Counter({'grounded': 172, 'memory': 78})
--------------------------------------------------
Phân bố sau oversampling:
  Train final: Counter({'grounded': 1031, 'memory': 906})
  Tổng mẫu train: 1937 (tăng thêm 453 mẫu memory)


In [12]:
import os
import zipfile
import torch

from google.colab import drive
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import PeftModel


if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")


BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
ZIP_PATH = (
    "/content/drive/MyDrive/raft_project/"
    "raft_qwen25_3b_lora.zip"
)
ADAPTER_DIR = "/content/raft_qwen25_3b_lora"


if not os.path.isfile(ZIP_PATH):
    raise FileNotFoundError(f"Không tìm thấy file ZIP: {ZIP_PATH}")


# Giải nén adapter vào local storage của Colab
os.makedirs(ADAPTER_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_file:
    zip_file.extractall(ADAPTER_DIR)

print(f"Đã giải nén vào: {ADAPTER_DIR}")
print(os.listdir(ADAPTER_DIR))

Đã giải nén vào: /content/raft_qwen25_3b_lora
['raft_qwen25_3b_lora']


In [13]:
# Nếu ZIP chứa một thư mục con, tìm thư mục thực sự chứa adapter_config.json
adapter_config_path = os.path.join(
    ADAPTER_DIR,
    "adapter_config.json",
)

if not os.path.isfile(adapter_config_path):
    candidates = []

    for root, _, files in os.walk(ADAPTER_DIR):
        if "adapter_config.json" in files:
            candidates.append(root)

    if not candidates:
        raise FileNotFoundError(
            "Không tìm thấy adapter_config.json sau khi giải nén."
        )

    ADAPTER_DIR = candidates[0]

print(f"Adapter directory: {ADAPTER_DIR}")


tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_DIR,
    local_files_only=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)


base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

base_model.config.pad_token_id = tokenizer.pad_token_id
base_model.config.eos_token_id = tokenizer.eos_token_id
base_model.config.use_cache = True


model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
    local_files_only=True,
)

model.eval()

print("Đã tải thành công model LoRA.")
print("Chat template:", tokenizer.chat_template is not None)

Adapter directory: /content/raft_qwen25_3b_lora/raft_qwen25_3b_lora


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Đã tải thành công model LoRA.
Chat template: True


## Bước 11 — Đánh giá

In [14]:
# Test model sau fine-tune bằng đúng instruction được truyền vào
SYSTEM_RAFT = """You are an expert medical AI assistant trained for retrieval-augmented QA.
You will receive multiple <DOCUMENT> blocks and a question.

The documents may contain distractors. The presence of a document does not mean that
it supports the answer. Inspect the documents independently.

Rules:

1. Ignore distractor documents. A document is relevant only if it explicitly states
the complete answer or provides enough information for one direct, medically standard
inference.

- If the documents only partially answer the question or merely mention related topics,
  they do not count as support. Use Memory Format instead.

2. Never mix formats. Every response must be strictly Grounded OR strictly Memory.

3. Grounded Format, used if and only if provided documents directly support the complete answer:

- `Evidence:` must be the first non-empty line.
- Include exactly one `Evidence:` section.
- Include at least one verbatim quote from a provided <DOCUMENT>.
- Wrap every quote with the literal tags `##begin_quote##` and `##end_quote##`.
- Each quote must be a contiguous substring of a provided <DOCUMENT>.
- Preserve the exact words and meaning of the source. Minor Unicode dash,
  quotation-mark, and whitespace differences are acceptable.
- Do not quote the instruction or the question.
- Include exactly one `Reasoning:` section explaining how the evidence answers the question.

4. Memory Format, used when no provided document supports the complete answer:

- `Reasoning:` must be the first non-empty line.
- Include exactly one `Reasoning:` section.
- Do NOT include `Evidence:`.
- Do NOT include `##begin_quote##` or `##end_quote##`.
- Do NOT claim that any provided document supports the answer.
- Use only established medical knowledge.
- Do not guess or invent mechanisms, numbers, citations, or specific details.

5. Formatting constraints:

- Do not add any text before the first required section.
- Do not use Markdown decoration around `Evidence:`, `Reasoning:`, or `<ANSWER>:`.
- Include exactly one `<ANSWER>:` tag in the entire output.
- The `<ANSWER>:` tag must appear strictly on the final line.
""".strip()

def test_finetuned_model(instruction: str, max_new_tokens: int = 512) -> str:
    # 1. Định dạng cấu trúc hội thoại ChatML chuẩn cho Qwen
    """
    Sinh output từ model LoRA đã fine-tune.
    Dùng đúng system prompt đã dùng lúc format training data.
    """
    messages = [
        {"role": "system", "content": SYSTEM_RAFT},
        {"role": "user", "content": instruction},
    ]

    # 2. Để tokenizer tự động đắp Chat Template chuẩn của mô hình
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1536).to(model.device)

    # Lấy ID của thẻ <|im_end|> để làm biển báo dừng cho Qwen
    im_end_token_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    eos_token_ids = [tokenizer.eos_token_id]

    if isinstance(im_end_token_id, int) and im_end_token_id >= 0:
        eos_token_ids.append(im_end_token_id)

    # --- Chuyển sang chế độ thi (Tắt học) và tắt tính đạo hàm ---
    model.eval()

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # Evaluation cần output xác định, có thể lặp lại
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=eos_token_ids,
        )

    # 4. Bỏ phần prompt, chỉ giữ phần mô hình tự sinh
    input_length = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0, input_length:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()

In [19]:
# ── Test trên vài sample ngẫu nhiên ────────────────────────────────────
import random
random.seed(42)

preview_samples = random.sample(
    test_samples,
    k=min(5, len(test_samples)),
)

for i, sample in enumerate(preview_samples, 1):
    result = test_finetuned_model(sample["instruction"])

    print("=" * 70)
    print(f"Sample {i}")
    print(f"Question: {sample['question']}")
    print(f"Gold answer_mode: {sample['answer_mode']}")
    print(f"\nModel answer:\n{result}")
    print(f"\nGold label:\n{sample['cot_answer'][:500]}")

Sample 1
Question: Which molecules and cellular processes are up‑regulated or activated by TNF, including adhesion molecules, prostaglandin E2, platelet‑activating factor, glucocorticoids, eicosanoids, and endothelial permeability?
Gold answer_mode: memory

Model answer:
**Reasoning:**  
TNF family members activate NFκB via TRAFs, leading to transcriptional activation of numerous genes. Among the up‑regulated or activated molecules are adhesion molecules (e.g., LFA‑1, ICAM‑1), prostaglandin E2, platelet‑activating factor, glucocorticoids, eicosanoids, and endothelial permeability.

<ANSWER>: TNF family members activate NFκB via TRAFs, resulting in transcriptional activation of adhesion molecules (e.g., LFA‑1, ICAM‑1), prostaglandin E2, platelet‑activating factor, glucocorticoids, eicosanoids, and endothelial permeability.

Gold label:
**Reasoning:**  
TNF acts on virtually all differentiated cells and triggers a broad inflammatory program. It induces transcription of genes encoding adh

## Bước 12: Metric

In [20]:
# Tự động cài đặt rouge-score nếu chưa có
import sys
try:
    import rouge_score
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rouge-score"])

import re
import unicodedata
from collections import Counter

import pandas as pd
from rouge_score import rouge_scorer
from tqdm.auto import tqdm


# ============================================================
# 1. REGEX VÀ CHUẨN HÓA DỮ LIỆU
# ============================================================

ANSWER_LINE_RE = re.compile(
    r"(?mi)^\s*<ANSWER>:\s*(\S.*)\s*$"
)

QUOTE_RE = re.compile(
    r"##begin_quote##\s*(.*?)\s*##end_quote##",
    flags=re.IGNORECASE | re.DOTALL,
)

DASHES_RE = re.compile(r"[\u2010\u2011\u2012\u2013\u2014\u2015\u2212\uFE58\uFE63\uFF0D]")
QUOTES_NORM_RE = re.compile(r"[\u2018\u2019\u201A\u201B\u201C\u201D\u201E\u201F]")

def normalize_text(text):
    """Chuẩn hóa để tính Token F1 theo chuẩn SQuAD."""
    text = unicodedata.normalize("NFKC", str(text or "")).casefold()
    text = "".join(
        char if unicodedata.category(char)[0] not in {"P", "S"} else " "
        for char in text
    )
    return " ".join(text.split())

def normalize_quote_whitespace(text):
    """
    Normalized Citation Match:
    Chuẩn hóa khoảng trắng, Unicode NFKC, và đồng nhất các biến thể
    dấu gạch ngang (dashes) / dấu ngoặc kép của OCR sách y khoa.
    """
    text = unicodedata.normalize("NFKC", str(text or ""))
    text = DASHES_RE.sub("-", text)
    text = QUOTES_NORM_RE.sub("\"", text)
    return re.sub(r"\s+", " ", text).strip()

def count_answer_tags(text):
    """Đếm tổng số lần xuất hiện của thẻ <ANSWER>: trong toàn bộ văn bản."""
    return len(
        re.findall(
            r"<ANSWER>:",
            str(text or ""),
            flags=re.IGNORECASE,
        )
    )

def extract_final_answer(text):
    """Trích xuất nội dung thẻ <ANSWER>: ở đầu dòng."""
    answers = ANSWER_LINE_RE.findall(str(text or ""))
    if len(answers) != 1:
        return ""
    return answers[0].strip()

def has_section(text, section_name):
    """
    Kiểm tra sự tồn tại của section Evidence: hoặc Reasoning:
    Chấp nhận cả trường hợp xuống dòng hoặc có text ngay sau dấu hai chấm.
    """
    return bool(
        re.search(
            rf"(?mi)^\s*{re.escape(section_name)}\s*:(?:\s*$|\s+\S)",
            str(text or ""),
        )
    )

def first_nonempty_line(text):
    """Lấy dòng không rỗng đầu tiên của output."""
    for line in str(text or "").splitlines():
        line = line.strip()
        if line:
            return line
    return ""

def last_nonempty_line(text):
    """Lấy dòng không rỗng cuối cùng của output."""
    for line in reversed(str(text or "").splitlines()):
        line = line.strip()
        if line:
            return line
    return ""

def starts_with_required_section(prediction, answer_mode):
    """
    Kiểm tra dòng đầu tiên theo quy định nghiêm ngặt của System Prompt:
    - Grounded: Phải bắt đầu bằng Evidence:
    - Memory: Phải bắt đầu bằng Reasoning:
    """
    first_line = first_nonempty_line(prediction)
    if answer_mode == "grounded":
        return first_line.startswith("Evidence:")
    return first_line.startswith("Reasoning:")

def ends_with_answer_tag(prediction):
    """Kiểm tra dòng cuối cùng có khớp toàn bộ với thẻ <ANSWER>: và có nội dung."""
    last_line = last_nonempty_line(prediction)
    return bool(re.fullmatch(r"(?i)<ANSWER>:\s*\S.*", last_line))

def token_f1(prediction, reference):
    pred_tokens = normalize_text(prediction).split()
    gold_tokens = normalize_text(reference).split()
    if not pred_tokens or not gold_tokens:
        return 0.0
    overlap = sum((Counter(pred_tokens) & Counter(gold_tokens)).values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(pred_tokens)
    recall = overlap / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

# ============================================================
# 2. LẤY TÀI LIỆU VÀ SO KHỚP TRÍCH DẪN (CITATION)
# ============================================================

def get_documents(sample):
    context = sample.get("context", {})
    if isinstance(context, str):
        return [context]

    def flatten_to_strings(value):
        if isinstance(value, str):
            return [value]
        if isinstance(value, list):
            documents = []
            for child in value:
                documents.extend(flatten_to_strings(child))
            return documents
        return []

    if isinstance(context, dict):
        documents = flatten_to_strings(context.get("sentences", []))
        if documents:
            return documents

    return re.findall(
        r"<DOCUMENT>\s*(.*?)\s*</DOCUMENT>",
        sample.get("instruction", ""),
        flags=re.IGNORECASE | re.DOTALL,
    )

def citation_accuracy(quotes, documents):
    """
    Đánh giá Normalized Citation Match:
    Quote phải là một chuỗi con liên tục trong ít nhất một tài liệu cung cấp.
    """
    if not quotes or not documents:
        return 0.0

    normalized_docs = [
        normalize_quote_whitespace(document)
        for document in documents
    ]

    correct_quotes = 0
    for quote in quotes:
        norm_quote = normalize_quote_whitespace(quote)
        if norm_quote and any(norm_quote in doc for doc in normalized_docs):
            correct_quotes += 1

    return correct_quotes / len(quotes)

# ============================================================
# 3. KIỂM TRA ĐỊNH DẠNG (FORMAT COMPLIANCE)
# ============================================================

def check_format_compliance(
    prediction,
    answer_mode,
    has_valid_answer_tag,
    quote_rule_ok,
):
    # 1. Thẻ answer hợp lệ và phải ở dòng cuối cùng (có nội dung)
    if not has_valid_answer_tag or not ends_with_answer_tag(prediction):
        return False

    # 2. Tuân thủ quy tắc quote (Grounded có quote; Memory tuyệt đối không có)
    if not quote_rule_ok:
        return False

    # 3. Dòng đầu tiên phải đúng thẻ section (không có text râu ria/chào hỏi trước đó)
    if not starts_with_required_section(prediction, answer_mode):
        return False

    # 4. Kiểm tra các section bên trong
    has_reasoning = has_section(prediction, "Reasoning")
    has_evidence = has_section(prediction, "Evidence")

    if answer_mode == "grounded":
        return has_evidence and has_reasoning
    else:
        return (not has_evidence) and has_reasoning

# ============================================================
# 4. HÀM ĐÁNH GIÁ MỘT OUTPUT
# ============================================================

def evaluate_raft_prediction(sample, prediction):
    prediction = str(prediction or "")

    answer_mode = sample.get("answer_mode")
    if answer_mode not in {"grounded", "memory"}:
        answer_mode = (
            "grounded"
            if sample.get("has_oracle_in_context")
            else "memory"
        )

    gold_answer = extract_final_answer(sample.get("cot_answer", ""))
    pred_answer = extract_final_answer(prediction)

    # Kiểm tra triệt để: Trong toàn bộ output chỉ được có đúng 1 tag <ANSWER>:
    answer_tag_count = count_answer_tags(prediction)
    has_valid_answer_tag = (answer_tag_count == 1 and bool(pred_answer))

    quotes = QUOTE_RE.findall(prediction)
    begin_count = prediction.lower().count("##begin_quote##")
    end_count = prediction.lower().count("##end_quote##")
    has_quote_marker = (begin_count > 0 or end_count > 0)

    quote_tags_well_formed = (
        begin_count == end_count == len(quotes)
        and all(bool(normalize_quote_whitespace(q)) for q in quotes)
    )

    if answer_mode == "grounded":
        quote_rule_ok = bool(quotes) and quote_tags_well_formed
        documents = get_documents(sample)
        citation_context = citation_accuracy(quotes, documents)
        citation_oracle = citation_accuracy(quotes, [sample.get("oracle_context", "")])
    else:
        quote_rule_ok = not has_quote_marker
        citation_context = None
        citation_oracle = None

    full_format_compliant = check_format_compliance(
        prediction=prediction,
        answer_mode=answer_mode,
        has_valid_answer_tag=has_valid_answer_tag,
        quote_rule_ok=quote_rule_ok,
    )

    # Citation compliance chỉ áp dụng cho Grounded; Memory để None để in ra N/A
    if answer_mode == "grounded":
        citation_compliant = (
            full_format_compliant
            and citation_context == 1.0
            and citation_oracle == 1.0
        )
    else:
        citation_compliant = None

    # Đo lường lỗi theo từng mode
    grounded_format_error = (
        (not full_format_compliant) if answer_mode == "grounded" else None
    )
    memory_quote_error = (
        has_quote_marker if answer_mode == "memory" else None
    )

    return {
        "id": sample["id"],
        "question": sample.get("question", ""),
        "answer_mode": answer_mode,

        # Metrics chất lượng câu trả lời
        "f1": token_f1(pred_answer, gold_answer),
        "rougeL": (
            rouge.score(gold_answer, pred_answer)["rougeL"].fmeasure
            if gold_answer and pred_answer else 0.0
        ),

        # Metrics định dạng
        "answer_tag_count": answer_tag_count,
        "has_answer_tag": has_valid_answer_tag,
        "has_quote": has_quote_marker,  # Phản ánh ý đồ trích dẫn thực tế của model
        "quote_rule_ok": quote_rule_ok,
        "has_evidence_section": has_section(prediction, "Evidence"),
        "has_reasoning_section": has_section(prediction, "Reasoning"),
        "full_format_compliant": full_format_compliant,

        # Metrics trích dẫn (Normalized Citation Match)
        "citation_accuracy_context": citation_context,
        "citation_accuracy_oracle": citation_oracle,
        "citation_compliant": citation_compliant,

        # Metrics phân tích lỗi theo mode
        "grounded_format_error": grounded_format_error,
        "memory_quote_error": memory_quote_error,

        # Dữ liệu đối chiếu
        "pred_answer": pred_answer,
        "gold_answer": gold_answer,
        "raw_prediction": prediction,
    }

# ============================================================
# 5. IN TỔNG KẾT BÁO CÁO KẾT QUẢ
# ============================================================

def print_raft_summary(metrics_df):
    def percent(series):
        series = series.dropna()
        if len(series) == 0:
            return "N/A"
        return f"{series.mean() * 100:.1f}%"

    def print_group(name, group):
        print(f"\n[{name}]")
        print(f"Rows                                       : {len(group)}")
        print(f"F1 final answer                            : {percent(group['f1'])}")
        print(f"ROUGE-L final answer                       : {percent(group['rougeL'])}")
        print(f"Valid <ANSWER>: format                     : {percent(group['has_answer_tag'])}")
        print(f"Quote marker                               : {percent(group['has_quote'])}")
        print(f"Correct quote rule                         : {percent(group['quote_rule_ok'])}")
        print(f"Evidence section                           : {percent(group['has_evidence_section'])}")
        print(f"Reasoning section                          : {percent(group['has_reasoning_section'])}")
        print(f"Full RAFT format compliance                : {percent(group['full_format_compliant'])}")
        print(f"Citation normalized match in input context : {percent(group['citation_accuracy_context'])}")
        print(f"Citation normalized match from oracle      : {percent(group['citation_accuracy_oracle'])}")
        print(f"Citation compliance                        : {percent(group['citation_compliant'])}")

        if name == "grounded":
            print(f"Grounded format error                      : {percent(group['grounded_format_error'])}")

        if name == "memory":
            print(f"Memory quote error                         : {percent(group['memory_quote_error'])}")

    print("\n" + "=" * 70)
    print("RAFT MODEL EVALUATION ON HELD-OUT TEST SET")
    print("=" * 70)

    print_group("overall", metrics_df)
    for mode in ["grounded", "memory"]:
        group = metrics_df[metrics_df["answer_mode"] == mode]
        if len(group) > 0:
            print_group(mode, group)

In [21]:
from tqdm import tqdm

# Đánh giá chất lượng dataset label trên test_samples
all_test_samples = list(test_samples)

print(f"Số sample trong test set: {len(all_test_samples)}")

# Lưu output thô theo id để có thể kiểm tra lại từng lỗi
predictions = {}

for sample in tqdm(all_test_samples, desc="Evaluating RAFT model"):
    predictions[sample["id"]] = test_finetuned_model(
        sample["instruction"],
        max_new_tokens=256,
    )

# Tính toàn bộ metric trên test set
rows = [
    evaluate_raft_prediction(
        sample,
        predictions[sample["id"]],
    )
    for sample in all_test_samples
]

metrics_df = pd.DataFrame(rows)

# In tổng kết
print_raft_summary(metrics_df)

# Lưu chi tiết để đọc các sample lỗi sau này
metrics_df.to_json(
    "raft_test_predictions_and_metrics.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)

print("\nĐã lưu: raft_test_predictions_and_metrics.jsonl")

Số sample trong test set: 250


Evaluating RAFT model: 100%|██████████| 250/250 [1:00:46<00:00, 14.59s/it]



RAFT MODEL EVALUATION ON HELD-OUT TEST SET

[overall]
Rows                                       : 250
F1 final answer                            : 48.9%
ROUGE-L final answer                       : 46.6%
Valid <ANSWER>: format                     : 85.6%
Quote marker                               : 58.4%
Correct quote rule                         : 84.4%
Evidence section                           : 58.0%
Reasoning section                          : 74.0%
Full RAFT format compliance                : 68.0%
Citation normalized match in input context : 70.9%
Citation normalized match from oracle      : 70.3%
Citation compliance                        : 68.6%

[grounded]
Rows                                       : 172
F1 final answer                            : 58.1%
ROUGE-L final answer                       : 56.3%
Valid <ANSWER>: format                     : 85.5%
Quote marker                               : 81.4%
Correct quote rule                         : 80.8%
Evidence section   